# Task 2: Image Recognition System

**Image recognition** is a widely used application of machine learning, where algorithms analyze digital images by measuring pixel values. It's employed in various contexts, including face recognition, which identifies individuals and can trigger relevant notifications.

## Technologies Used
- **OpenCV** — Computer vision and image processing
- **NumPy** — Numerical operations on pixel arrays
- **scikit-learn** — Machine learning classification
- **Matplotlib / Seaborn** — Data visualization
- **Flask** — Web application backend

---

## 1. Setup & Imports

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import os

# Display settings
plt.style.use('dark_background')
sns.set_theme(style='darkgrid')
%matplotlib inline

print(f'OpenCV version: {cv2.__version__}')
print(f'NumPy version: {np.__version__}')

## 2. Image Loading & Basic Operations

Images are represented as multi-dimensional NumPy arrays. Each pixel has values for Blue, Green, and Red channels (BGR in OpenCV).

In [ ]:
# Create a sample test image with shapes
def create_test_image(width=400, height=300):
    img = np.zeros((height, width, 3), dtype=np.uint8)
    # Background gradient
    for i in range(height):
        img[i, :] = [int(50 + i * 0.3), int(20 + i * 0.2), int(100 - i * 0.1)]
    # Draw shapes
    cv2.circle(img, (100, 100), 50, (0, 255, 200), -1)
    cv2.rectangle(img, (200, 50), (300, 150), (255, 100, 50), -1)
    cv2.ellipse(img, (300, 220), (60, 40), 30, 0, 360, (100, 200, 255), -1)
    cv2.putText(img, 'Image Recognition', (50, 270),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    return img

test_img = create_test_image()

# Display
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB))
axes[0].set_title('Original Image (RGB)', fontsize=12)
axes[0].axis('off')

gray = cv2.cvtColor(test_img, cv2.COLOR_BGR2GRAY)
axes[1].imshow(gray, cmap='gray')
axes[1].set_title('Grayscale', fontsize=12)
axes[1].axis('off')
plt.tight_layout()
plt.show()

print(f'Image Shape: {test_img.shape}')
print(f'Data Type: {test_img.dtype}')
print(f'Total Pixels: {test_img.shape[0] * test_img.shape[1]:,}')

## 3. Face Detection with Haar Cascades

Haar Cascade Classifiers are machine learning-based detectors trained on positive and negative images. OpenCV provides pre-trained cascades for faces, eyes, and smiles.

In [ ]:
# Load Haar Cascade classifiers
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)
eye_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_eye.xml'
)
smile_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_smile.xml'
)

print('Cascade classifiers loaded successfully!')
print(f'Available cascades in OpenCV: {len(os.listdir(cv2.data.haarcascades))} files')

In [ ]:
def detect_faces_in_image(img):
    """Detect faces, eyes, and smiles in an image."""
    result = img.copy()
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    faces = face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(30, 30))
    print(f'Faces detected: {len(faces)}')
    
    for i, (x, y, w, h) in enumerate(faces):
        cv2.rectangle(result, (x, y), (x+w, y+h), (0, 255, 128), 2)
        cv2.putText(result, f'Face {i+1}', (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 128), 2)
        
        roi_gray = gray[y:y+h, x:x+w]
        roi_color = result[y:y+h, x:x+w]
        
        eyes = eye_cascade.detectMultiScale(roi_gray, 1.1, 5)
        for (ex, ey, ew, eh) in eyes[:2]:
            center = (ex + ew//2, ey + eh//2)
            cv2.circle(roi_color, center, max(ew,eh)//2, (255, 200, 0), 2)
        print(f'  Face {i+1}: {len(eyes)} eyes detected')
    
    return result, faces

# Create a synthetic face-like image for demonstration
face_img = np.ones((300, 300, 3), dtype=np.uint8) * 200
cv2.ellipse(face_img, (150, 150), (80, 100), 0, 0, 360, (180, 160, 140), -1)
cv2.circle(face_img, (120, 130), 12, (50, 50, 50), -1)  # Left eye
cv2.circle(face_img, (180, 130), 12, (50, 50, 50), -1)  # Right eye
cv2.ellipse(face_img, (150, 185), (25, 10), 0, 0, 180, (50, 50, 100), 2)  # Mouth

result, faces = detect_faces_in_image(face_img)
plt.figure(figsize=(8, 4))
plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
plt.title(f'Face Detection Result ({len(faces)} faces)', fontsize=14)
plt.axis('off')
plt.show()

## 4. Color Analysis with K-Means Clustering

K-Means clustering groups similar pixel colors together to find the **dominant colors** in an image. This is an unsupervised machine learning technique.

In [ ]:
def extract_dominant_colors(img, k=5):
    """Extract k dominant colors using K-Means clustering."""
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    pixels = rgb.reshape(-1, 3).astype(np.float32)
    
    if len(pixels) > 10000:
        indices = np.random.choice(len(pixels), 10000, replace=False)
        pixels = pixels[indices]
    
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 1.0)
    _, labels, centers = cv2.kmeans(
        pixels, k, None, criteria, 10, cv2.KMEANS_PP_CENTERS
    )
    
    _, counts = np.unique(labels, return_counts=True)
    percentages = counts / len(labels) * 100
    
    # Sort by percentage
    order = np.argsort(-percentages)
    centers = centers[order]
    percentages = percentages[order]
    
    return centers.astype(int), percentages

# Analyze test image colors
colors, pcts = extract_dominant_colors(test_img, k=5)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Show color swatches
swatch = np.zeros((50, 300, 3), dtype=np.uint8)
x_start = 0
for color, pct in zip(colors, pcts):
    x_end = x_start + int(pct * 3)
    swatch[:, x_start:x_end] = color
    x_start = x_end
ax1.imshow(swatch)
ax1.set_title('Dominant Color Palette', fontsize=12)
ax1.axis('off')

# Bar chart
bar_colors = [f'#{c[0]:02x}{c[1]:02x}{c[2]:02x}' for c in colors]
ax2.barh(range(len(pcts)), pcts, color=bar_colors, edgecolor='white', linewidth=0.5)
ax2.set_xlabel('Percentage (%)')
ax2.set_ylabel('Color')
ax2.set_title('Color Distribution', fontsize=12)
ax2.set_yticks(range(len(pcts)))
ax2.set_yticklabels([f'#{c[0]:02x}{c[1]:02x}{c[2]:02x}' for c in colors])
ax2.invert_yaxis()

plt.tight_layout()
plt.show()

## 5. Edge Detection

Edge detection identifies boundaries in images where pixel intensity changes rapidly. We use **Canny**, **Sobel**, and **Laplacian** algorithms.

In [ ]:
gray = cv2.cvtColor(test_img, cv2.COLOR_BGR2GRAY)

# Different edge detection methods
canny = cv2.Canny(gray, 50, 150)
sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
sobel = np.sqrt(sobelx**2 + sobely**2).astype(np.uint8)
laplacian = np.uint8(np.abs(cv2.Laplacian(gray, cv2.CV_64F)))

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
titles = ['Original', 'Canny', 'Sobel', 'Laplacian']
images = [gray, canny, sobel, laplacian]

for ax, img, title in zip(axes, images, titles):
    ax.imshow(img, cmap='gray')
    ax.set_title(title, fontsize=12)
    ax.axis('off')

plt.suptitle('Edge Detection Methods Comparison', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 6. Image Filters Gallery

OpenCV provides numerous image transformation and filter capabilities.

In [ ]:
def apply_filters(img):
    """Apply multiple filters and return results."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    filters = {}
    
    filters['Grayscale'] = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    filters['Blur'] = cv2.GaussianBlur(img, (21, 21), 0)
    
    kernel = np.array([[-1,-1,-1],[-1,9,-1],[-1,-1,-1]])
    filters['Sharpen'] = cv2.filter2D(img, -1, kernel)
    
    sepia_k = np.array([[0.272,0.534,0.131],[0.349,0.686,0.168],[0.393,0.769,0.189]])
    filters['Sepia'] = np.clip(cv2.transform(img, sepia_k), 0, 255).astype(np.uint8)
    
    inv = 255 - gray
    blur = cv2.GaussianBlur(inv, (21, 21), 0)
    filters['Sketch'] = cv2.cvtColor(cv2.divide(gray, 255 - blur, scale=256), cv2.COLOR_GRAY2BGR)
    
    filters['Invert'] = cv2.bitwise_not(img)
    
    return filters

filter_results = apply_filters(test_img)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, (name, fimg) in zip(axes.flat, filter_results.items()):
    ax.imshow(cv2.cvtColor(fimg, cv2.COLOR_BGR2RGB))
    ax.set_title(name, fontsize=11)
    ax.axis('off')

plt.suptitle('Image Filters Gallery', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 7. Histogram Analysis

Color histograms represent the distribution of pixel intensities across each color channel (R, G, B).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Original image
axes[0].imshow(cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB))
axes[0].set_title('Original Image', fontsize=12)
axes[0].axis('off')

# RGB Histogram
channel_colors = {'Blue': ('b', 0), 'Green': ('g', 1), 'Red': ('r', 2)}
for name, (color, idx) in channel_colors.items():
    hist = cv2.calcHist([test_img], [idx], None, [256], [0, 256])
    axes[1].plot(hist, color=color, label=name, alpha=0.7)
    axes[1].fill_between(range(256), hist.flatten(), alpha=0.15, color=color)

axes[1].set_title('RGB Color Histogram', fontsize=12)
axes[1].set_xlabel('Pixel Intensity')
axes[1].set_ylabel('Frequency')
axes[1].legend()
axes[1].set_xlim([0, 256])

plt.tight_layout()
plt.show()

## 8. Image Classification with scikit-learn

Using **HOG (Histogram of Oriented Gradients)** features with an SVM classifier for simple image classification.

In [ ]:
from skimage.feature import hog
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Generate synthetic training data (shapes)
def generate_shape_dataset(n_per_class=100, size=64):
    images, labels = [], []
    for _ in range(n_per_class):
        # Circle
        img = np.zeros((size, size), dtype=np.uint8)
        r = np.random.randint(10, 25)
        cx, cy = np.random.randint(r+2, size-r-2, 2)
        cv2.circle(img, (cx, cy), r, 255, -1)
        images.append(img); labels.append(0)
        
        # Rectangle
        img = np.zeros((size, size), dtype=np.uint8)
        x1, y1 = np.random.randint(5, 25, 2)
        x2, y2 = x1 + np.random.randint(15, 30), y1 + np.random.randint(15, 30)
        cv2.rectangle(img, (x1, y1), (min(x2, size-2), min(y2, size-2)), 255, -1)
        images.append(img); labels.append(1)
        
        # Triangle
        img = np.zeros((size, size), dtype=np.uint8)
        pts = np.array([[32, 10], [10, 54], [54, 54]]) + np.random.randint(-5, 5, (3, 2))
        cv2.fillPoly(img, [pts], 255)
        images.append(img); labels.append(2)
    
    return np.array(images), np.array(labels)

X_imgs, y = generate_shape_dataset(n_per_class=200)

# Extract HOG features
X_hog = np.array([hog(img, pixels_per_cell=(8, 8), cells_per_block=(2, 2)) for img in X_imgs])

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X_hog, y, test_size=0.2, random_state=42)

# Train SVM
clf = SVC(kernel='rbf', C=10, gamma='scale')
clf.fit(X_train, y_train)

# Evaluate
y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'Classification Accuracy: {acc:.2%}\n')
print(classification_report(y_test, y_pred, target_names=['Circle', 'Rectangle', 'Triangle']))

In [ ]:
# Visualize some predictions
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
class_names = ['Circle', 'Rectangle', 'Triangle']
indices = np.random.choice(len(X_test), 10, replace=False)

for ax, idx in zip(axes.flat, indices):
    # Find original image index
    orig_idx = np.where((X_hog == X_test[idx]).all(axis=1))[0][0]
    ax.imshow(X_imgs[orig_idx], cmap='gray')
    pred = y_pred[list(range(len(X_test))).index(list(range(len(X_test)))[list(indices).index(idx)])]
    true = y_test[list(indices)[list(indices).index(idx)]]
    color = 'lime' if pred == true else 'red'
    ax.set_title(f'{class_names[pred]}', color=color, fontsize=10)
    ax.axis('off')

plt.suptitle(f'Shape Classification Results (Accuracy: {acc:.1%})', fontsize=14)
plt.tight_layout()
plt.show()

## 9. Contour Detection & Analysis

In [ ]:
gray = cv2.cvtColor(test_img, cv2.COLOR_BGR2GRAY)
edges = cv2.Canny(gray, 50, 150)
contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

result = test_img.copy()
for i, cnt in enumerate(contours):
    color = tuple(int(c) for c in cv2.applyColorMap(
        np.uint8([[int(i * 255 / max(len(contours), 1))]]), cv2.COLORMAP_HSV)[0][0])
    cv2.drawContours(result, [cnt], -1, color, 2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.imshow(cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB))
ax1.set_title('Original')
ax1.axis('off')
ax2.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
ax2.set_title(f'Contours Detected: {len(contours)}')
ax2.axis('off')
plt.tight_layout()
plt.show()

## 10. Summary

This notebook demonstrated key image recognition techniques:

| Technique | Algorithm | Library |
|---|---|---|
| Face Detection | Haar Cascades | OpenCV |
| Color Extraction | K-Means Clustering | OpenCV |
| Edge Detection | Canny, Sobel, Laplacian | OpenCV |
| Image Filters | Convolution Kernels | OpenCV |
| Histogram Analysis | calcHist | OpenCV |
| Shape Classification | HOG + SVM | scikit-learn |
| Contour Detection | findContours | OpenCV |

### Web Application
Run the web app with:
```bash
python app.py
```
Then visit **http://localhost:5000** for the full interactive interface.